# Recommendation system models

### Libraries Import

In [22]:
import sqlite3
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

### Dataset Import

##### Products Dataset

In [2]:
conn = sqlite3.connect("../../amazon_electronics.db")
products_df = pd.read_sql_query("SELECT * FROM products", conn)
conn.close()
# 
products_df.head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating_count,about_product,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,399.0,1.099,0.64,24269,High Compatibility : Compatible With iPhone 12...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,199.0,349.000,0.43,43994,"Compatible with all Type C enabled devices, be...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,199.0,1.899,0.90,7928,【 Fast Charger& Data Sync】-With built-in safet...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,329.0,699.000,0.53,94363,The boAt Deuce USB 300 2 in 1 cable is compati...,https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,154.0,399.000,0.61,16905,[CHARGE & SYNC FUNCTION]- This cable comes wit...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


##### Ratings Dataset

In [3]:
conn = sqlite3.connect("../../amazon_electronics.db")

ratings_df = pd.read_sql_query(
    "SELECT user_id, product_id, rating FROM ratings WHERE rating IS NOT NULL",
    conn
)

conn.close()

ratings_df.head()

,user_id,product_id,rating
0,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...",B07JW9H4J1,4.2
1,"AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...",B098NS6PVG,4.0
2,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...",B096MSW6CT,3.9
3,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...",B08HDJ86NZ,4.2
4,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...",B08CF3B7N1,4.2


In [4]:
ratings_df.info()
ratings_df["rating"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1464 entries, 0 to 1463
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   user_id     1464 non-null   object 
 1   product_id  1464 non-null   object 
 2   rating      1464 non-null   float64
dtypes: float64(1), object(2)
memory usage: 34.4+ KB


count    1464.000000
mean        4.096585
std         0.291674
min         2.000000
25%         4.000000
50%         4.100000
75%         4.300000
max         5.000000
Name: rating, dtype: float64

### Non personalized recommender

In [5]:
merged_df = pd.merge(
    products_df,
    ratings_df,
    on="product_id"
)

top_products = (
    merged_df
    .groupby(
        ['product_id', 'product_name', 'discounted_price']
    )['rating']
    .mean()
    .reset_index(name='avg_rating')
    .sort_values(by='avg_rating', ascending=False)
    .head(5)
)

top_products['avg_rating'] = top_products['avg_rating']

top_products

,product_id,product_name,discounted_price,avg_rating
1348,B0BQRJ3C47,"REDTECH USB-C to Lightning Cable 3.3FT, [Apple...",249.000,5.0
1120,B09ZHCJDP1,Amazon Basics Wireless Mouse | 2.4 GHz Connect...,499.000,5.0
1341,B0BP7XLX48,Syncwire LTG to USB Cable for Fast Charging Co...,399.000,5.0
1347,B0BQ3K23Y1,"Oratech Coffee Frother electric, milk frother ...",279.000,4.8
1204,B0B53DS4TF,"Instant Pot Air Fryer, Vortex 2QT, Touch Contr...",4.995,4.8


### Content based filtering

In [6]:
# ── 1. Category encoding ────────────────────────────
products_df["category_split"] = (
    products_df["category"]
    .str.split(r"[|&]")
    .apply(lambda x: list(set(x)))
)
mlb = MultiLabelBinarizer()
category_matrix = csr_matrix(mlb.fit_transform(products_df["category_split"]))

# ── 2. Price scaling ─────────────────────────────────
scaler = MinMaxScaler()
price_scaled = csr_matrix(
    scaler.fit_transform(products_df[["discounted_price"]])
)

# ── 3. TF-IDF on product_name ─────────────────────────────────────────────────
tfidf_name = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),   # unigrams + bigrams catch "fast charging", "Type-C", etc.
    max_features=500
)
name_matrix = tfidf_name.fit_transform(
    products_df["product_name"].fillna("")
)

# ── 4. TF-IDF on about_product ────────────────────────────────────────────────
tfidf_about = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=1000     # richer descriptions → more features
)
about_matrix = tfidf_about.fit_transform(
    products_df["about_product"].fillna("")
)

In [7]:
# ── 5. Weighted feature matrix ────────────────────────────────────────────────
W_CATEGORY = 2.0
W_PRICE    = 0.5
W_NAME     = 1.5
W_ABOUT    = 1.0

feature_matrix = hstack([
    category_matrix * W_CATEGORY,
    price_scaled    * W_PRICE,
    name_matrix     * W_NAME,
    about_matrix    * W_ABOUT,
])

# ── 6. Similarity matrix ──────────────────────────────────────────────────────
similarity_matrix = cosine_similarity(feature_matrix)

In [8]:
# função para obter os produtos mais semelhantes a um produto específico
def get_nearest_products(product_id, similarity_matrix, products_df, k):
    product_index_map = pd.Series(
        products_df.index,
        index=products_df["product_id"]
    ).to_dict()

    if product_id not in product_index_map:
        return {}

    index = product_index_map[product_id]

    similarity_scores = list(enumerate(similarity_matrix[index]))
                                                                                

    similarity_scores = sorted(      
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    top_products = similarity_scores[1:k+1]

    k_nearest = {
        products_df.iloc[i]["product_id"]: score
        for i, score in top_products
    }

    return k_nearest

In [9]:
get_nearest_products(
    "B07YTNKVJQ",
    similarity_matrix,
    products_df,
    k=5
)

{'B09X79PP8F': np.float64(0.9435025251581326),
 'B0B4HKH19N': np.float64(0.9176682182685089),
 'B08DDRGWTJ': np.float64(0.916310464659329),
 'B083342NKJ': np.float64(0.9062356054012883),
 'B01GGKZ0V6': np.float64(0.9019895012768593)}

In [10]:
rated_produnts = {'B0789LZTCJ': 4.2, 'B094JNXNPV': 3.5}

def get_recommendations(rated_products, n):     

    candidates = {}

    for p, r in rated_products.items():

        k_nearest = get_nearest_products(p,similarity_matrix,products_df,n*2)

        for product, cos_sim in k_nearest.items():
            if product in rated_products:
                continue
            if product in candidates:
                candidates[product] += float(cos_sim)  
            else:                                      
                candidates[product] = float(cos_sim) * r
    
    recommendations = [
        product_id 
        for product_id, _ in sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:n]
    ]

    return recommendations


get_recommendations(rated_produnts, 5)



['B09PNR6F8Q', 'B082LZGK39', 'B07CRL2GY6', 'B09NHVCHS9', 'B08WRWPM22']

### Colaborative filtering

Train/Test split

In [11]:
ratings_df = ratings_df.drop_duplicates(
    subset=["user_id", "product_id"]
)

In [12]:
# remover duplicados (1464 → 1360 ratings únicos)
ratings_df = ratings_df.drop_duplicates(subset=["user_id", "product_id"])

# split por utilizador: users com ≥2 ratings têm 1 entrada no test set
train_list, test_list = [], []

for user_id, group in ratings_df.groupby("user_id"):
    if len(group) < 2:
        train_list.append(group)
    else:
        test_sample  = group.sample(n=1, random_state=42)
        train_sample = group.drop(test_sample.index)
        train_list.append(train_sample)
        test_list.append(test_sample)

train_df = pd.concat(train_list).reset_index(drop=True)
test_df  = pd.concat(test_list).reset_index(drop=True)

In [13]:
print("Total ratings :", len(ratings_df))
print("Train ratings :", len(train_df))
print("Test ratings  :", len(test_df))
print("Users dataset :", ratings_df["user_id"].nunique())
print("Users treino  :", train_df["user_id"].nunique())
print("Users teste   :", test_df["user_id"].nunique())

# confirmar que não há users no teste que não estão no treino
missing = set(test_df["user_id"]) - set(train_df["user_id"])
print("Users no teste sem histórico de treino:", len(missing))

Total ratings : 1360
Train ratings : 1265
Test ratings  : 95
Users dataset : 1193
Users treino  : 1193
Users teste   : 95
Users no teste sem histórico de treino: 0


In [14]:
users_train = set(train_df["user_id"])
users_test = set(test_df["user_id"])

missing_users = users_test - users_train

print("Users no teste que não existem no treino:", len(missing_users))

Users no teste que não existem no treino: 0


Criar user-item matrix com train

In [15]:
# user-item matrix (apenas dados de treino)
user_item_matrix        = train_df.pivot_table(
    index="user_id", columns="product_id", values="rating"
)
user_item_matrix_filled = user_item_matrix.fillna(0)

# item-item similarity
item_similarity    = cosine_similarity(user_item_matrix_filled.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

# user-user similarity com mean-centering (reduz bias de utilizadores que
# tendem a dar ratings altos ou baixos sistematicamente)
user_means         = user_item_matrix.mean(axis=1)
user_item_centered = user_item_matrix.sub(user_means, axis=0).fillna(0)
user_similarity    = cosine_similarity(user_item_centered)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("user_item_matrix   :", user_item_matrix.shape)
print("item_similarity_df :", item_similarity_df.shape)
print("user_similarity_df :", user_similarity_df.shape)


user_item_matrix   : (1193, 1260)
item_similarity_df : (1260, 1260)
user_similarity_df : (1193, 1193)


Item-based Função de recomendações

In [16]:
def get_cf_recommendations(user_id, n=5):
    if user_id not in user_item_matrix.index:
        return []

    user_ratings = user_item_matrix.loc[user_id].dropna()

    scores = {}

    for product_id, rating in user_ratings.items():
        similar_items = item_similarity_df[product_id]

        for sim_product, sim_score in similar_items.items():
            if sim_product == product_id:
                continue

            scores[sim_product] = scores.get(sim_product, 0) + sim_score * rating

    # remover produtos já avaliados
    scores = {
        k: v for k, v in scores.items()
        if k not in user_ratings.index
    }

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [item for item, _ in ranked[:n]]

In [17]:
#testar
user_id = train_df["user_id"].iloc[0]

get_cf_recommendations(user_id)

['B002PD61Y4', 'B002SZEOLG', 'B003B00484', 'B003L62T7W', 'B004IO5BMQ']

User-based Função de recomendações

In [18]:
# 2. Função de recomendação user-based
def get_user_based_cf_recommendations(user_id, n=5, k_users=10):
    """
    Para cada utilizador semelhante (top-k), agrega as suas ratings
    ponderadas pela similaridade. Exclui produtos já avaliados pelo user.
    """
    if user_id not in user_item_matrix.index:
        return []

    # Utilizadores mais semelhantes (excluindo o próprio)
    sim_users = (
        user_similarity_df[user_id]
        .drop(index=user_id)
        .sort_values(ascending=False)
        .head(k_users)
    )

    already_rated = set(
        user_item_matrix.loc[user_id].dropna().index
    )

    scores = {}

    for sim_user_id, sim_score in sim_users.items():
        # Produtos que este utilizador semelhante avaliou
        sim_user_ratings = user_item_matrix.loc[sim_user_id].dropna()

        for product_id, rating in sim_user_ratings.items():
            if product_id in already_rated:
                continue
            scores[product_id] = scores.get(product_id, 0) + sim_score * rating

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [product_id for product_id, _ in ranked[:n]]

In [19]:
# 3. Teste
user_id = train_df["user_id"].iloc[0]

print("User-based CF:", get_user_based_cf_recommendations(user_id))

User-based CF: ['B09HSKYMB3', 'B00HVXS7WC', 'B09XJ1LM7R', 'B09V2Q4QVQ', 'B09YDFDVNS']


Avaliar modelos collaborative filtering

In [23]:
def predict_rating_item_based(user_id, product_id, k=10):
    if user_id not in user_item_matrix.index:
        return None
    if product_id not in item_similarity_df.columns:
        return None

    user_ratings = user_item_matrix.loc[user_id].dropna()
    if user_ratings.empty:
        return None

    sim_scores = (
        item_similarity_df[product_id]
        .reindex(user_ratings.index)
        .dropna()
    )
    sim_scores = sim_scores[sim_scores > 0]
    if sim_scores.empty:
        return None

    top_k   = sim_scores.sort_values(ascending=False).head(k)
    ratings = user_ratings.reindex(top_k.index)
    denom   = top_k.sum()
    if denom == 0:
        return None
    return (top_k * ratings).sum() / denom


def predict_rating_user_based(user_id, product_id, k=10):
    if user_id not in user_similarity_df.index:
        return None
    if product_id not in user_item_matrix.columns:
        return None

    sim_users = (
        user_similarity_df[user_id]
        .drop(index=user_id)
        .sort_values(ascending=False)
    )
    sim_users = sim_users[sim_users > 0]
    if sim_users.empty:
        return None

    ratings_for_product = user_item_matrix[product_id].dropna()
    if ratings_for_product.empty:
        return None

    valid_users = sim_users.index.intersection(ratings_for_product.index)
    if len(valid_users) == 0:
        return None

    top_users = sim_users.loc[valid_users].head(k)
    ratings   = ratings_for_product.loc[top_users.index]
    denom     = top_users.sum()
    if denom == 0:
        return None
    return (top_users * ratings).sum() / denom


# ── loop de avaliação ──────────────────────────────────────────────────────────
item_preds, item_actuals = [], []
user_preds, user_actuals = [], []

for _, row in test_df.iterrows():
    uid, pid, actual = row["user_id"], row["product_id"], row["rating"]

    p_item = predict_rating_item_based(uid, pid)
    if p_item is not None:
        item_preds.append(p_item)
        item_actuals.append(actual)

    p_user = predict_rating_user_based(uid, pid)
    if p_user is not None:
        user_preds.append(p_user)
        user_actuals.append(actual)

# ── resultados ─────────────────────────────────────────────────────────────────
print("RESULTADOS")

if item_preds:
    rmse_item = np.sqrt(mean_squared_error(item_actuals, item_preds))
    print(f"Item-based CF  →  RMSE: {rmse_item:.4f}  ({len(item_preds)}/{len(test_df)} pares)")
else:
    print("Item-based CF  →  sem previsões suficientes.")

if user_preds:
    rmse_user = np.sqrt(mean_squared_error(user_actuals, user_preds))
    print(f"User-based CF  →  RMSE: {rmse_user:.4f}  ({len(user_preds)}/{len(test_df)} pares)")
else:
    print("User-based CF  →  0 previsões.")
    print("  Causa: dataset esparso (média 1.14 ratings/user).")
    print("  O modelo item-based é usado na integração do sistema.")


RESULTADOS
Item-based CF  →  sem previsões suficientes.
User-based CF  →  0 previsões.
  Causa: dataset esparso (média 1.14 ratings/user).
  O modelo item-based é usado na integração do sistema.


In [24]:
# ── Avaliação CF por Hit Rate@K ────────────────────────────────────────────────

def hit_rate_at_k(rec_fn, test_df, k=10):
    """
    Para cada user no test set, verifica se o item held-out
    aparece nas top-K recomendações geradas pelo modelo.
    """
    hits  = 0
    total = 0

    for _, row in test_df.iterrows():
        uid            = row["user_id"]
        held_out_item  = row["product_id"]

        recommendations = rec_fn(uid, n=k)

        if not recommendations:
            continue

        total += 1
        if held_out_item in recommendations:
            hits += 1

    if total == 0:
        return 0.0, 0

    return hits / total, total


# ── Resultados ─────────────────────────────────────────────────────────────────
for k in [5, 10, 20]:
    hr_item, n_item = hit_rate_at_k(get_cf_recommendations, test_df, k=k)
    hr_user, n_user = hit_rate_at_k(get_user_based_cf_recommendations, test_df, k=k)

    print(f"K={k:2d} | "
          f"Item-based Hit Rate: {hr_item:.4f} ({n_item} users) | "
          f"User-based Hit Rate: {hr_user:.4f} ({n_user} users)")

K= 5 | Item-based Hit Rate: 0.0000 (95 users) | User-based Hit Rate: 0.0000 (95 users)
K=10 | Item-based Hit Rate: 0.0000 (95 users) | User-based Hit Rate: 0.0000 (95 users)
K=20 | Item-based Hit Rate: 0.0000 (95 users) | User-based Hit Rate: 0.0000 (95 users)


In [25]:
# Verificar se as funções geram recomendações (ou retornam listas vazias)
sample_users = test_df["user_id"].iloc[:5]

for uid in sample_users:
    item_recs = get_cf_recommendations(uid, n=10)
    user_recs = get_user_based_cf_recommendations(uid, n=10)
    held_out  = test_df[test_df["user_id"] == uid]["product_id"].values[0]

    print(f"\nUser: {uid[:30]}...")
    print(f"  Held-out item : {held_out}")
    print(f"  Item-based recs ({len(item_recs)}): {item_recs[:3]}")
    print(f"  User-based recs ({len(user_recs)}): {user_recs[:3]}")


User: AE27UOZENYSWCQVQRRUQIV2ZM7VA,A...
  Held-out item : B09V2PZDX8
  Item-based recs (10): ['B002PD61Y4', 'B002SZEOLG', 'B003B00484']
  User-based recs (10): ['B09HSKYMB3', 'B07ZR4S1G4', 'B00HVXS7WC']

User: AE2JTMRKTUOIVIZWS2WDGTMNTU4Q,A...
  Held-out item : B084N133Y7
  Item-based recs (10): ['B002PD61Y4', 'B002SZEOLG', 'B003B00484']
  User-based recs (10): ['B09HSKYMB3', 'B07ZR4S1G4', 'B00HVXS7WC']

User: AE3XH7AL52IBMYH77L5KO4DGTCDA,A...
  Held-out item : B07YY1BY5B
  Item-based recs (10): ['B002PD61Y4', 'B002SZEOLG', 'B003B00484']
  User-based recs (10): ['B09HSKYMB3', 'B07ZR4S1G4', 'B00HVXS7WC']

User: AEC6UDCEAUIBIFHGQDQ4KR67GC4A,A...
  Held-out item : B0BF54972T
  Item-based recs (10): ['B002PD61Y4', 'B002SZEOLG', 'B003B00484']
  User-based recs (10): ['B09HSKYMB3', 'B07ZR4S1G4', 'B00HVXS7WC']

User: AECPFYFQVRUWC3KGNLJIOREFP5LQ,A...
  Held-out item : B098NS6PVG
  Item-based recs (10): ['B002PD61Y4', 'B002SZEOLG', 'B003B00484']
  User-based recs (10): ['B09HSKYMB3', 'B07ZR4S